# Prioritize unlabeled data for labeling

Use the [`prioritization`](../reference/presets.md#prioritization) workflow to rank incoming data by what to label
next, given an already-labeled reference dataset and a trained model.

You are a model developer or data scientist who manages an active-learning or labeling pipeline and needs to
prioritize unlabeled data under budget constraints.

Prioritization operates during data acquisition. Given reference data and a trained model, it ranks unlabeled inputs
to target novel or challenging cases before retraining. Run it after {doc}`data_coverage` has shown which concepts
your data misses. To ask how much labeled data your model needs, continue with {doc}`measure_data_sufficiency`. See
{doc}`../concepts/Prioritization` for background on ranking policies.

## What you will do

- Load MilitaryVehicles and split it into labeled reference and unlabeled pools.
- Train a classifier on the labeled pool covering 20 vehicle types, holding out 4 Air Defense systems.
- Inject corrupted and duplicate frames into the unlabeled pool to test automated pruning.
- Execute the `prioritization` workflow to rank unlabeled samples.
- Measure ranking efficiency against random selection baselines and model uncertainty sampling.

## What you will learn

- How to configure and execute `prioritization` with `run_task()`.
- How reference datasets and extractor representations direct prioritization.
- How `hard_first` ordering prioritizes out-of-distribution or challenging samples.
- How to benchmark prioritization gains against random sampling baselines.
- Why model uncertainty sampling fails to detect novel classes and how distance-based prioritization resolves this.
- How pre-prioritization pruning removes outliers and duplicates from the ranking pool.

## Prerequisites

- Install `dataeval-flow`, which includes `dataeval`, `datamaite` and `torch`, and install `torchvision`.
- Install `maite-datasets[datamaite]` to download and export MilitaryVehicles.
- Make sure you have network access for the initial dataset download.

## Lay out the data root

The data root is the folder you run this notebook from. After a first run it holds:

```text
project/
├── data/
│   ├── militaryvehicles/                    # downloaded on the first run; the export is made from it
│   └── militaryvehicles_datamaite_train/    # exported on the first run; read: both pools are views of it
├── models/
│   └── vehiclenet.pt                        # written: the trained classifier; read back by the extractor
└── cache/                                   # written: embeddings and image statistics, reused on a re-run
```

The notebook builds its pipeline in Python and reads no configuration file. The config's `model_path`,
`models/vehiclenet.pt`, is relative to the data root. The ResNet-18 weights the classifier starts from download the
first time into torchvision's own cache, outside this folder.

## Build a labeled pool and an unlabeled pool

[MilitaryVehicles](https://huggingface.co/datasets/leibnitz-lab/military_vehicles) contains 9,444 images across 24
vehicle types. The dataset includes coarse categories such as tanks, BMPs, BTRs, self-propelled artillery, and air
defense systems.

In this tutorial, you will hold out the four Air Defense systems (`30N6E`, `Iskander`, `Pantsir-S1`, `Rs-24`) to
simulate encountering an unannotated operational category:

- The labeled pool has 4,000 frames drawn from the 20 known vehicle types.
- The unlabeled pool has 1,500 frames drawn from the remaining data. It contains leftover known types and the held-out
  Air Defense systems.

In [ ]:
from pathlib import Path

import numpy as np
from dataeval.data import Filter, Indices, Limit, Shuffle, View
from datamaite import load_ic
from maite_datasets.image_classification import MilitaryVehicles

from dataeval_flow.config import PreprocessingStep

data_root = Path("./data")
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

vehicles = load_ic(data_root / "militaryvehicles_datamaite_train" / "train", dataset_format="huggingface_vision")
index2label = vehicles.metadata.get("index2label", {})

HELD_OUT_NAMES = {"30N6E", "Iskander", "Pantsir-S1", "Rs-24"}
held_out = {i for i, name in index2label.items() if name in HELD_OUT_NAMES}
known_classes = [i for i in sorted(index2label) if i not in held_out]

# Both pools are constructed using View operations.
# You build them in Python here because the classification model trains on the
# labeled pool directly before pipeline execution.
#
# Labeled pool: The twenty known classes only.
labeled_dataset = View(vehicles, [Filter(classes=known_classes), Shuffle(seed=42), Limit(4000)])
labeled_indices = labeled_dataset.resolve_indices()

# Unlabeled pool: The remaining frames, containing Air Defense systems and leftover known types.
pool_dataset = View(vehicles, [Indices(labeled_indices, exclude=True), Shuffle(seed=7), Limit(1500)])

pool_labels = np.array([int(np.argmax(pool_dataset[i][1])) for i in range(len(pool_dataset))])
n_novel = int(np.isin(pool_labels, sorted(held_out)).sum())
print(f"Held out (Air Defense): {sorted(index2label[i] for i in held_out)}")
print(f"Labeled pool:   {len(labeled_dataset)} frames, {len(known_classes)} types")
print(
    f"Unlabeled pool: {len(pool_dataset)} frames, {n_novel} of them Air Defense "
    f"({n_novel / len(pool_dataset) * 100:.0f}%)"
)

### Inject corrupted and duplicate images into the test data

Simulate incoming data corruption with an in-memory wrapper that corrupts a slice of the test set on the fly:

| Indices   | Corruption            |
|-----------|-----------------------|
| 900–919   | Duplicates of 200–219 |
| 920–939   | Gaussian blur         |
| 940–959   | Random noise          |
| 960–979   | Overexposure          |
| 980–999   | Underexposure         |

In [ ]:
from collections.abc import Mapping
from typing import Any

import numpy as np
from numpy.typing import NDArray
from PIL import Image, ImageFilter


class CorruptedTestDataset:
    """Wraps a MAITE dataset and injects synthetic corruptions into indices 900-999.

    Labels are left untouched throughout, so ground truth stays available for
    verification even though the pixel content at those indices is corrupted.
    """

    _DUPLICATE_OFFSET = 700  # maps index i in [900, 920) to source index i - 700

    def __init__(self, dataset: Any) -> None:
        self._dataset = dataset

    def __len__(self) -> int:
        return len(self._dataset)

    def __getitem__(self, index: int) -> tuple[NDArray[Any], Any, Mapping[str, Any]]:
        if 900 <= index < 920:
            image, _, _ = self._dataset[index - self._DUPLICATE_OFFSET]
            _, target, metadata = self._dataset[index]
            return np.array(image, copy=True), target, metadata

        image, target, metadata = self._dataset[index]

        if 920 <= index < 940:
            image = self._blur(image)
        elif 940 <= index < 960:
            image = self._add_noise(image, seed=index)
        elif 960 <= index < 980:
            image = self._adjust_exposure(image, 100)
        elif 980 <= index < 1000:
            image = self._adjust_exposure(image, -200, halve=True)

        return image, target, metadata

    @staticmethod
    def _blur(image: NDArray[Any]) -> NDArray[Any]:
        hwc = np.transpose(image, (1, 2, 0))  # CHW -> HWC
        blurred = Image.fromarray(hwc, mode="RGB").filter(ImageFilter.GaussianBlur(radius=3))
        return np.transpose(np.array(blurred, dtype=image.dtype), (2, 0, 1))  # back to CHW

    @staticmethod
    def _add_noise(image: NDArray[Any], *, seed: int) -> NDArray[Any]:
        arr = image.astype(np.int16)
        noise = np.random.default_rng(seed).integers(-80, 80, arr.shape, dtype=np.int16)
        return np.clip(arr + noise, 0, 255).astype(image.dtype)

    @staticmethod
    def _adjust_exposure(image: NDArray[Any], delta: int, *, halve: bool = False) -> NDArray[Any]:
        arr = np.clip(image.astype(np.int16) + delta, 0, 255)
        if halve:
            arr = arr // 2
        return arr.astype(image.dtype)


test_dataset = CorruptedTestDataset(pool_dataset)

print(
    "Test data corruption wrapper ready: 20 exact duplicates (900-919), "
    "20 blurred (920-939), 20 noisy (940-959), "
    "20 bright (960-979), 20 dark (980-999)"
)

## Train a classifier on the labeled pool

Use a feature extractor trained on your specific label space so that the embeddings reflect domain characteristics.

You fine-tune a linear head on top of a frozen pretrained ResNet-18 trunk. You extract embeddings from the
128-dimensional `embed` layer connecting the trunk and the classification head.

In [ ]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F  # noqa: N812
import torchvision

random.seed(42)
torch.manual_seed(42)


class VehicleNet(nn.Module):
    """A frozen pretrained trunk, a learned embedding, and a head over known classes.

    ``embed`` bundles Linear + ReLU so a forward hook captures the activated output:
    the representation the classifier was trained on.
    """

    def __init__(self, num_classes: int) -> None:
        super().__init__()
        trunk = torchvision.models.resnet18(weights="IMAGENET1K_V1")
        trunk.fc = nn.Identity()
        self.trunk = trunk
        self.embed = nn.Sequential(nn.Linear(512, 128), nn.ReLU())
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.embed(self.trunk(x)))


model = VehicleNet(len(known_classes))
for parameter in model.trunk.parameters():
    parameter.requires_grad = False
model.trunk.eval()

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def _trunk_features(dataset: Any) -> tuple[torch.Tensor, list[int]]:
    """Run the frozen trunk once over a dataset to generate training features for the head."""
    out: list[torch.Tensor] = []
    labels: list[int] = []
    with torch.no_grad():
        for start in range(0, len(dataset), 64):
            batch = []
            for i in range(start, min(start + 64, len(dataset))):
                image, target, _ = dataset[i]
                labels.append(int(np.argmax(target)))
                img = torch.tensor(np.asarray(image), dtype=torch.float32) / 255.0
                img = F.interpolate(img.unsqueeze(0), size=(224, 224), mode="bilinear", align_corners=False)
                batch.append((img.squeeze(0) - _MEAN) / _STD)
            out.append(model.trunk(torch.stack(batch)))
    return torch.cat(out), labels

In [ ]:
_remap = {c: k for k, c in enumerate(known_classes)}
features, labeled_labels = _trunk_features(labeled_dataset)
targets = torch.tensor([_remap[c] for c in labeled_labels], dtype=torch.long)

head = nn.Sequential(model.embed, model.classifier)
optimizer = torch.optim.Adam(head.parameters(), lr=1e-3)
for _epoch in range(40):
    optimizer.zero_grad()
    loss = F.cross_entropy(head(features), targets)
    loss.backward()
    optimizer.step()
model.eval()

accuracy = (head(features).argmax(1) == targets).float().mean().item()
print(f"Head trained: loss={loss.item():.3f}, accuracy on the labeled pool={accuracy * 100:.1f}%")

### Save the trained model to disk

Save the model so `TorchExtractorConfig` can load it and hook the `embed` layer. See {doc}`../how_to/torch_embeddings`
for how to hook an intermediate layer of a network.

In [ ]:
model_path = Path("./models/vehiclenet.pt")
model_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model, str(model_path))
print(f"Model saved to {model_path}")

## Build the workflow configuration

You configure these parts:

| Part | Configuration |
|---|---|
| Reference | The labeled pool (4,000 frames, 20 vehicle types). |
| Incoming data | The unlabeled pool (1,500 frames, including held-out Air Defense and corrupted samples). |
| Extractor | The trained VehicleNet hooking the `embed` layer for 128-dimensional embeddings. |
| Method | KNN with `hard_first` ordering to rank samples farthest from reference neighbors. |
| Pruning | Outlier and exact-duplicate detection to filter invalid samples before ranking. |

The `prioritization` preset ranks and does not prune. A custom workflow prunes the reference and the pool with
[`outliers`](../reference/evaluators.md#outliers), [`duplicates`](../reference/evaluators.md#duplicates) and
[`remove`](../reference/transforms.md#remove) steps, then runs the preset as its step `rank`, on what they kept.

In [ ]:
from dataeval_flow.config import (
    DatasetProtocolConfig,
    PipelineConfig,
    PreprocessorConfig,
    SourceConfig,
    TaskConfig,
)
from dataeval_flow.config.extractors import TorchExtractorConfig
from dataeval_flow.evaluators.quality import DuplicatesConfig, OutliersConfig
from dataeval_flow.steps import CustomWorkflowConfig, StepEntry
from dataeval_flow.workflows.prioritization import PrioritizationWorkflowConfig

ref_dataset = labeled_dataset

# The adaptive bound is lower than the default 3.5, to catch subtler corruptions.
outliers = OutliersConfig(name="outliers", flags=["dimension", "pixel", "visual"], outlier_threshold=("adaptive", 3.0))
duplicates = DuplicatesConfig(name="duplicates", merge_near_duplicates=True)
ranking = PrioritizationWorkflowConfig(
    name="ranking", prioritization={"method": "knn", "k": 5, "order": "hard_first", "policy": "difficulty"}
)


def pruned(prefix: str, source: str) -> list[StepEntry]:
    """`source` without its outliers and its exact duplicates: steps `outliers-<prefix>`, `duplicates-<prefix>` and
    `<prefix>-clean`."""
    plans = {f"duplicates-{prefix}": {"dup_types": ["exact"], "keep": "first"}, f"outliers-{prefix}": {"min_flags": 1}}
    return [
        StepEntry(name=f"outliers-{prefix}", evaluator="outliers", input=source),
        StepEntry(name=f"duplicates-{prefix}", evaluator="duplicates", input=source),
        StepEntry(name=f"{prefix}-clean", transform="remove", input=source, plans=plans),
    ]


workflow = CustomWorkflowConfig(
    name="vehicles_prioritize",
    inputs=["reference", {"name": "pools", "list": True}],
    steps=[
        *pruned("reference", "reference"),
        *pruned("pool", "pools"),
        StepEntry(name="rank", workflow="ranking", input=["reference-clean", "pool-clean"]),
    ],
)

task = TaskConfig(
    name="prioritize_pool",
    workflow="vehicles_prioritize",
    sources=["ref_src", "test_src"],
    extractor="cnn_extractor",
)

### Assemble the pipeline config

The first source is the reference dataset. Each later source is unlabeled data to prioritize. `TorchExtractorConfig`
points to the saved model and hooks the `embed` layer.

Include a preprocessor that resizes frames to 224x224 and normalizes pixel values using ImageNet statistics, to match
the model training conditions.

In [ ]:
config = PipelineConfig(
    max_processes=2,
    datasets=[
        DatasetProtocolConfig(name="ref_ds", dataset=ref_dataset),
        DatasetProtocolConfig(name="test_ds", dataset=test_dataset),
    ],
    preprocessors=[
        PreprocessorConfig(
            name="imagenet",
            steps=[
                PreprocessingStep(step="Resize", params={"size": [224, 224], "antialias": True}),
                PreprocessingStep(step="ToDtype", params={"dtype": "float32", "scale": True}),
                PreprocessingStep(
                    step="Normalize",
                    params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
                ),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="ref_src", dataset="ref_ds"),
        SourceConfig(name="test_src", dataset="test_ds"),
    ],
    extractors=[
        TorchExtractorConfig(
            name="cnn_extractor",
            model_path=str(model_path),
            layer_name="embed",
            preprocessor="imagenet",
            batch_size=32,
        ),
    ],
    evaluators=[outliers, duplicates],
    workflows=[ranking, workflow],
    tasks=[task],
)

## Run the prioritization workflow

Call `run_task()` to prune outliers and rank unlabeled samples. To run prioritization pipelines in a container, see
{doc}`../how_to/containerized_workflows`.

In [ ]:
from dataeval_flow import run_task

result = run_task(config, task, cache_dir=Path("./cache"))

In [ ]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

### Prioritization report

The report lists each step of the workflow. Its `rank/prioritization` step pictures the 25 highest-priority and the 25
lowest-priority frames. See {doc}`../how_to/read_evaluation_outputs` for how to read ranking reports and result
envelopes.

In [ ]:
print(result.report())

### View the HTML report

`result.to_html()` renders the same report as one HTML page. It holds everything the text report holds, laid out for
reading on screen, with tables you can sort and filter. The page follows your system's dark mode setting, so here it
may not match this site's theme. See {doc}`view_html_reports` for the rest of the page.

In [ ]:
# The page styles its whole document, so the docs show it in an iframe of its own.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [ ]:
show(result.to_html())

## Inspect what pruning removed

The pruning phase detects outliers and duplicates in both reference and incoming data. The `reference-clean` and
`pool-clean` steps record how many frames they dropped, and the ranking holds only the pool frames `pool-clean` kept.
Check whether pruning removed the injected corrupted samples.

In [ ]:
pool_clean = result.steps["pool-clean"].elements["test_src"]
reference_clean = result.steps["reference-clean"]
print(f"Removed from the reference: {reference_clean.details['removed']['items']} frames")
print(f"Removed from the pool:      {pool_clean.details['removed']['items']} frames")

In [ ]:
# Which pool frames were pruned? `pool-clean` holds the frames it kept, by their index in the pool.
kept = pool_clean.output.resolve_indices()
all_test_indices = set(range(len(test_dataset)))
pruned_indices = sorted(all_test_indices - set(kept))
print(f"Pool frames: {len(all_test_indices)} total, {len(kept)} clean, {len(pruned_indices)} pruned")

# Check overlap with known corrupted ranges
corrupted_ranges = {
    "duplicates (900-919)": set(range(900, 920)),
    "blurred (920-939)": set(range(920, 940)),
    "noisy (940-959)": set(range(940, 960)),
    "bright (960-979)": set(range(960, 980)),
    "dark (980-999)": set(range(980, 1000)),
}
pruned_set = set(pruned_indices)
print("\nOverlap with injected corruptions:")
for name, indices in corrupted_ranges.items():
    overlap = pruned_set & indices
    print(f"  {name}: {len(overlap)}/{len(indices)} pruned")

planted = set(range(900, 1000)) | set(range(200, 220))
other_pruned = pruned_set - planted
if other_pruned:
    print(f"  Other frames pruned: {len(other_pruned)} of {len(all_test_indices) - len(planted)}")

Pruned frames are excluded from the final prioritization ranking so labeling budgets target clean data.

## Benchmark the ranking against random selection

Benchmark the prioritized ranking against random selection. This evaluation measures the proportion of held-out Air
Defense samples retrieved across various labeling budgets.

In [ ]:
# `selected` holds the pool in ranked order. Its indices count within the cleaned pool, so map them back through
# `pool-clean`'s to index the pool itself.
selected = result.steps["rank/selected"].elements["test_src"].output
top_indices = [kept[i] for i in selected.resolve_indices()]

# `pool_labels` was read straight off the view; the corruption wrapper alters pixels only,
# so a frame's ground-truth label is the same whether or not it was corrupted.
is_novel = np.isin(pool_labels[top_indices], sorted(held_out))
baseline_pct = 100.0 * is_novel.mean()

print(f"Baseline (share of the ranked set): {baseline_pct:.0f}% Air Defense\n")
print(f"{'Budget':>8}  {'Air Defense':>12}  {'vs random':>10}")
for n in (50, 100, 200, 500):
    pct = 100.0 * is_novel[:n].mean()
    print(f"{n:>8}  {pct:>11.0f}%  {pct / baseline_pct:>9.2f}x")

The prioritized ranking exceeds the random baseline across budgets. The relative advantage is greatest at small
budgets (such as the top 50 to 100 samples) and gradually converges toward the baseline as larger fractions of the
pool are labeled.

In [ ]:
cumulative = np.cumsum(is_novel)
n_items = np.arange(1, len(cumulative) + 1)
novel_pct = 100.0 * cumulative / n_items

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(n_items, novel_pct, linewidth=2, label="Prioritized (hard_first)")
    ax.axhline(baseline_pct, color="gray", linestyle="--", linewidth=1, label=f"Random baseline ({baseline_pct:.0f}%)")
    ax.set_xlabel("Labeling budget (top-N frames)")
    ax.set_ylabel("% Air Defense")
    ax.set_title("Novel-category concentration across the priority ranking")
    ax.set_xlim(1, len(cumulative))
    ax.set_ylim(0, max(100, novel_pct[:50].max() * 1.2))
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    for n in (50, 100, 200, 500, len(cumulative)):
        print(f"Top {n:4d}: {novel_pct[n - 1]:.0f}% novel")

### Compare with ranking by model uncertainty

Evaluate entropy-based uncertainty sampling against distance-based prioritization. Compute the entropy of predicted
class probabilities across the unlabeled pool:

In [ ]:
def _normalized(idx: int) -> torch.Tensor:
    """One pool frame, in the shape the model expects."""
    img = torch.tensor(np.asarray(test_dataset[idx][0]), dtype=torch.float32) / 255.0
    img = F.interpolate(img.unsqueeze(0), size=(224, 224), mode="bilinear", align_corners=False)
    return (img.squeeze(0) - _MEAN) / _STD


# Batched, because the whole pool at 224x224 does not fit in memory at once.
_probs = []
with torch.no_grad():
    for start in range(0, len(top_indices), 32):
        batch = torch.stack([_normalized(i) for i in top_indices[start : start + 32]])
        _probs.append(torch.softmax(model(batch), dim=1))
probabilities = torch.cat(_probs).numpy()

entropy = -(probabilities * np.log(probabilities + 1e-9)).sum(axis=1)
by_entropy = np.argsort(-entropy)
novel_by_entropy = is_novel[by_entropy]

print(f"{'Budget':>8}  {'hard_first':>11}  {'by entropy':>11}  {'baseline':>9}")
for n in (50, 100, 200, 500):
    print(
        f"{n:>8}  {100.0 * is_novel[:n].mean():>10.0f}%  "
        f"{100.0 * novel_by_entropy[:n].mean():>10.0f}%  {baseline_pct:>8.0f}%"
    )

Uncertainty sampling fails to prioritize novel classes because classifiers assign confident predictions to unfamiliar
inputs matching known feature patterns. Uncertainty sampling identifies samples near known decision boundaries,
whereas distance-based prioritization identifies samples outside known clusters.

## Visualize the top-ranked images (optional)

Plot the top-ranked prioritized samples to inspect flagged imagery:

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        idx = top_indices[i]
        img, target, _ = test_dataset[idx]
        img_arr = np.transpose(np.asarray(img), (1, 2, 0))  # CHW -> HWC
        label = int(np.argmax(target))
        ax.imshow(img_arr)
        ax.set_title(f"#{i + 1} idx={idx}\nlabel={label}", fontsize=9)
        ax.axis("off")
    fig.suptitle("Top 10 prioritized images (hard_first)", fontsize=13)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib to visualize: pip install matplotlib")

## Keep the top of the ranking

Set `select: {n: ...}` or `select: {fraction: ...}` on the `ranking` entry and each pool's top items become
`rank.selected`, a Dataset later steps can read. Here you keep the top 100 frames for a labeling batch.

In [ ]:
from dataeval_flow.workflows.prioritization import SelectSettings

top_100 = config.model_copy(
    update={"workflows": [ranking.model_copy(update={"select": SelectSettings(n=100)}), workflow]}
)
batch = run_task(top_100, task, cache_dir=Path("./cache")).steps["rank/selected"].elements["test_src"].output
print(f"Selected for labeling: {len(batch)} frames, the first {len(batch)} of the ranking")

## Conclusion

In this tutorial, you learned how to:

- Train a custom embedding extractor on reference data.
- Configure the `prioritization` workflow with KNN distance metrics and `hard_first` ordering.
- Prune outliers and exact duplicates with steps of a custom workflow, then run the preset as its step.
- Run the preset via `run_task()`.
- Benchmark prioritization results against random selection baselines.
- Contrast distance-based prioritization with model uncertainty sampling.
- Keep the top of each ranking as `selected`, with `select.n` or `select.fraction`.

## Next steps

- {doc}`measure_data_sufficiency` — Estimate how much training data a model needs.
- {doc}`../concepts/Prioritization` — Learn how the ranking methods and ordering policies work.
- {doc}`../how_to/read_evaluation_outputs` — Read ranking reports and result envelopes.

## On your own

Repeat the analysis with your own data, model or settings:

- Set `dataset` on the two `DatasetProtocolConfig` entries to your own labeled reference dataset and your own
  unlabeled pool.
- Set `model_path` on `TorchExtractorConfig` to your own saved model, and `layer_name` to the layer whose output you
  want as embeddings. It is `embed` here.
- Change the `size` of the `Resize` step and the `mean` and `std` of the `Normalize` step in the `imagenet`
  preprocessor to match your model.
- Set `method` in the `prioritization` settings of the `ranking` entry to `kmeans_distance`, `kmeans_complexity`,
  `hdbscan_distance` or `hdbscan_complexity`. It is `knn` here, and `k` sets its number of neighbors.
- Set `order` to `easy_first` to rank prototypical samples first. It is `hard_first` here.
- Set `policy` to `class_balanced` to balance the ranking across known classes. It is `difficulty` here.
- Change `outlier_threshold` and `flags` on the `outliers` entry to control the pruning sensitivity.
- Prune with a `quality` step on each input instead of the `outliers`, `duplicates` and `remove` steps. It also
  removes near duplicates.
- Change `n` in `SelectSettings`, which is 100 here, or set `fraction` instead, to keep a different batch size.